In [ ]:
import re
import string
import unicodedata
import pandas as pd
import spacy as scy
from joblib import load
from itertools import product
from unidecode import unidecode
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

## Passo 1) Carregar o modelo1

O modelo1 inicial será utilizado para realização da predição dos tokens dos relatos. A saída desse modelo será utilizada para obtenção dos dados de entrada para o modelo2.

In [ ]:
rf_e2 = load('rf_exp2.joblib')

## Passo 2) Carregar o modelo Corretor

In [ ]:
rf_corretor = load('rf_corretor.joblib')

## Passo 3) Carregar os dados para predição

In [ ]:
base_relato = pd.read_csv('base_para_teste_final.csv')

In [ ]:
base_relato.shape

## Passo 4) Criar lista de saída

### Passo 4.1) Pré-processamento dos relatos

Aqui são realizadas duas etapas a primeira o pré-processamento dos relatos, onde vamos adequar os relatos para ser a entrada do modelo1 removendo stopwors, caracteres especiais e etc...

### Passo 4.2) Realização da predição dos dados com o modelo1

A partir dos dados pré-processados, entramos na etapa de predição utilizando o modelo1.

Após a predição é retornado um dafaframe contendo os dados rotulados. Para cada relato é inserido um caracter coringa que será explicado logo a frente.

In [ ]:
######################################################################################################################################################
# Pré-Processamento dos relatos
######################################################################################################################################################
def preprocess(relato):
    def remove_stop_words(relato):
        # Remove stop words em português
        sw = stopwords.words('portuguese')
        sw = [unidecode(word).upper() for word in sw]
        return ' '.join([token for token in re.split("[, ]", relato) if len(token) > 2 and token not in sw])
    
    def clean_relato(relato):
        # Remoção de números e pontuação
        relato = re.sub(r'(<[^<>]*>)|(&NBSP;)|(&nbsp;)', " ", relato)
        relato = re.sub(r"[^\w\s]", " ", relato)
        relato = re.sub(r"[\d]", " ", relato)
        relato = remove_stop_words(relato)
        return relato
    
    relato = clean_relato(relato)
    return relato

def normalize_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.upper()

print("Iniciando pré-processamento...")
base_relato['relato'] = base_relato['relato'].apply(normalize_text)
base_relato['relato_tratado'] = base_relato['relato'].apply(preprocess)
print("Pré-processamento finalizado!")
######################################################################################################################################################
# Predição utilizando o modelo1
######################################################################################################################################################
def loadOutputList(df_bop : pd.DataFrame):
    rf_exp_2 = load('rf_exp2.joblib')

    output_list = []

    for text in df_bop['relato_tratado']:
        text = text.split()
        df = pd.DataFrame(text, columns=['palavra'])

        df['y_palavra'] = rf_exp_2.predict(df['palavra'].to_list())

        for _, row in df.iterrows():
            output_list.append({
                'palavra': row['palavra'],
                'y_palavra': int(row['y_palavra']),
            })
        output_list.append({
                'palavra': '#',
                'y_palavra': 2,
            })
    
    output_list.pop()

    return pd.DataFrame(output_list)
print("Carregando predição do modelo...")
output_list = loadOutputList(base_relato)
print("Base de dados de predição criada!")
######################################################################################################################################################

essa é a estrutura gerada após a predição do modelo1

In [ ]:
output_list

### Estratégia utilizada para separação dos textos no dataframe abaixo:

É evidente que os relatos não estão dispostos em texto corrido.  

Qual o passo a passo?  

Passo 1) Pré-processamento: Aqui vamos remover stopwords, caracteres especiais e etc 
Logo, após isso vamos realizar a tokenização do relato, transformando o memso em uma vetor de tokens que será armazenada em um dataframe.  

Passo 2) Inserção do delimitador utilizando caracter coringa '#' : Após a inserção de cada vetor de tokens um caractere coringa será inserido ao final do processo de inserção. Este por sua vez tem o papel de delimitar o inicio e fim de um documento de texto tokenizado.

Após a criação da lista de saída (output_list) é realizada a validação da quantidade de '#', verificando se a mesma corresponde a quantidade de relatos na base

In [ ]:
output_list[output_list['palavra'] == '#']

## Passo 5) Criar lista temporária

- Para cada item, realizar a predição com o modelo do João
- Inserir a predição na lista de saída.
- Se a predição for negativa, inserir a predição na lista temporária, guardando o indice na lista de saída.

In [ ]:
def vector_input_corretor(output_list : pd.DataFrame):
    
    def LoadPedSucWord(output_list : pd.DataFrame):
        list_aux_psw = []
        df_aux = output_list.copy()
        print(df_aux)

        df_aux['predecessor'] = df_aux['palavra'].shift(1).fillna('#')
        df_aux['y_predecessor'] = df_aux['predecessor'].apply(
            lambda x: df_aux[df_aux['palavra'] == x]['y_palavra'].iloc[0] if x != '#' else 2
        )

        df_aux['sucessor'] = df_aux['palavra'].shift(-1).fillna('#')
        df_aux['y_sucessor'] = df_aux['sucessor'].apply(
            lambda x: df_aux[df_aux['palavra'] == x]['y_palavra'].iloc[0] if x != '#' else 2
        )

        print(df_aux)

        for _, row in df_aux.iterrows():
            list_aux_psw.append({
                'predecessor': row['predecessor'],
                'y_predecessor': int(row['y_predecessor']),
                'palavra': row['palavra'],
                'y_palavra': int(row['y_palavra']),
                'sucessor': row['sucessor'],
                'y_sucessor': int(row['y_sucessor']),
            })

        return pd.DataFrame(df_aux)

    print("Realizando coleta de predecessores e sucessores...")
    temp_list = LoadPedSucWord(output_list)
    temp_list = temp_list[temp_list['y_palavra'] == 0].reset_index()
    indices_originais = temp_list['index']
    print("Predecessores e Sucessores coletados!")

    def bigrams_vector(lista_tokens):
        vectorizer = load('vectorizer.joblib')
        alfabeto = list(string.ascii_uppercase)
        alfabeto_bigrams = [''.join(bigram) for bigram in list(product(alfabeto, repeat=2))]
        vector = vectorizer.transform(lista_tokens)
        return pd.DataFrame(vector.todense(), columns=alfabeto_bigrams)

    print("Criando vetor de bigramas para os tokens...")
    token = temp_list['palavra']
    tokens_vector_bigrams = bigrams_vector(token.to_list())
    print("Vetores de bigramas dos tokens criados!")
    print("Concatenando vetor de bigramas com dados de predecessores e sucessores dos tokens...")
    tokens_vector_bigrams = pd.concat([tokens_vector_bigrams, temp_list[['predecessor','y_predecessor', 'palavra', 'y_palavra','sucessor', 'y_sucessor']]], axis=1)
    print("Concatenação concluída!")

    def create_base_cg(lista_tokens):
        nlp = scy.load('pt_core_news_lg')
        def renomear(doc):
            mapeamento_pos = {
                'NOUN': '<SUBSTANTIVO>',
                'PROPN': '<SUBSTANTIVO>',
                'VERB': '<VERBO>',
                'AUX': '<VERBO>',
                'ADJ': '<ADJETIVO>',
                'ADV': '<ADVERBIO>',
                'PUNCT': '<PONTUACAO>',
                'CCONJ': '<CONJUNCAO>',
                'SCONJ': '<CONJUNCAO>',
                'PRON': '<PRONOME>',
                'DET': '<ARTIGO>',
                'ADP': '<PREPOSIÇÃO>',
                'NUM': '<NUMERAL>',
                'SYM': '<SIMBOLO>',
                'X': '<PALAVRA ESTRANGEIRA>',
                'SPACE': '<ESPACO>',
            }
            
            return [mapeamento_pos.get(token.pos_, token.pos_) for token in doc]


        word_class = []
        for token in lista_tokens:
            if token == '#':
                word_class.append('<NULL>')
            else:
                lreview = token.lower()
                ldoc = nlp(lreview)
                classes = renomear(ldoc)
                for i in range(len(token.split())):
                    word_class.append(classes[i])

        return pd.DataFrame({'palavra':lista_tokens, 'classe':word_class})

    print("Realizando classificação gramatical dos predecessores, sucessores e dos tokens...")
    tokens_vector_bigrams['cg_p'] = create_base_cg(tokens_vector_bigrams['predecessor'])['classe']
    tokens_vector_bigrams['cg_w'] = create_base_cg(tokens_vector_bigrams['palavra'])['classe']
    tokens_vector_bigrams['cg_s'] = create_base_cg(tokens_vector_bigrams['sucessor'])['classe']
    print("Classificação gramatical realizada com sucesso!")
    def transformLabelsCG(tokens_vector_bigrams : pd.DataFrame):
        lb = LabelEncoder()

        labels = ['<SUBSTANTIVO>', '<VERBO>', '<ADJETIVO>', '<ADVERBIO>', '<PONTUACAO>',
                '<CONJUNCAO>', '<PRONOME>', '<ARTIGO>', '<PREPOSIÇÃO>', '<NUMERAL>',
                '<SIMBOLO>', '<PALAVRA ESTRANGEIRA>', '<ESPACO>', '<NULL>']

        lb.fit(labels)
        
        tokens_vector_bigrams['cg_p'] = lb.transform(tokens_vector_bigrams['cg_p'])
        tokens_vector_bigrams['cg_w'] = lb.transform(tokens_vector_bigrams['cg_w'])
        tokens_vector_bigrams['cg_s'] = lb.transform(tokens_vector_bigrams['cg_s'])

        return tokens_vector_bigrams
    print("Realizando a transformação dos labels da classificação gramatical para entrada adequada do modelo...")
    tokens_vector_bigrams = transformLabelsCG(tokens_vector_bigrams)
    print("Transformação dos labels da classificação gramatical realizada com sucesso!")
    print("Removendo as colunas contendo os tokens dos predecessores, sucessores e palavras...")
    final_tokens_vector_bigrams = tokens_vector_bigrams.drop(['y_palavra'], axis=1)
    print("Remoção realizada com sucesso!")
    print("Retornando vetor de entrada para o modelo.")

    return final_tokens_vector_bigrams, indices_originais

In [ ]:
test_vector_input_model, indices_originais = vector_input_corretor(output_list)

In [ ]:
vector_input_temp_list = test_vector_input_model.drop(columns=['palavra', 'predecessor', 'sucessor'])

In [ ]:
vector_input_temp_list

In [ ]:
vector_input_temp_list.dtypes

In [ ]:
result = rf_corretor.predict(vector_input_temp_list)

In [ ]:
temp_list = pd.DataFrame({'y_pred':result})
temp_list = pd.concat([temp_list, indices_originais], axis=1)

In [ ]:
temp_list

## CAPTURANDO OS NOVOS Falsos Positivos DO MODELO 2

carregando os falsos positivos do modelo 1

In [ ]:
gab = pd.read_csv('gabarito_corretor.csv')

In [ ]:
fp_m1 = pd.read_csv('M1_falsos_positivos.csv')
fp_m1

carrgando os falsos positivos do modelo dois

In [ ]:
fp_m2 = pd.read_csv('M2_falsos_positivos.csv')
fp_m2

Aqui realizo a verificação dos FALSOS POSITIVOS que existem dentro do modelo 1 e que  também fazem parte do modelo 2.


Aqui é realizado isso pra trazer apenas os tokens que surgiram a partir do modelo 2, para verificação do que o modelo 2 utilizou para a predição.

Assim podemos tentar identificar padroes para justificativas dos novos FALSOS POSITIVOS

In [ ]:
fp_m2_novos = fp_m2[~fp_m2['palavra'].isin(fp_m1['palavra'])]
fp_m2_novos.to_csv("M2_falsos_positivos.csv", index=False)

Etapa de captura dos vetores de entrada do modelo 2

Afim de não realizar todo o processamento novamente para explicação dos falsos positivos. Foi feito.

PASSO 1) Carregamento do vetor de entrada no modelo 2

In [ ]:
vector_input_temp_list

PASSO 2) CRIAÇÃO DO VETOR DE BIGRAMAS DOS FALSOS POSITIVOS DO MODELO 2

In [ ]:
def bigrams_vector(lista_tokens):
    vectorizer = load('vectorizer.joblib')

    alfabeto = list(string.ascii_uppercase)
    alfabeto_bigrams = [''.join(bigram) for bigram in product(alfabeto, repeat=2)]

    vector = vectorizer.transform(lista_tokens)

    bigrams_df = pd.DataFrame(vector.todense(), columns=alfabeto_bigrams)
    
    return bigrams_df


salvando a palavra

In [ ]:
token = fp_m2_novos['palavra']
tokens_vector_bigrams = bigrams_vector(token.to_list())
tokens_vector_bigrams

In [ ]:
alfabeto = list(string.ascii_uppercase)
alfabeto_bigrams = [''.join(bigram) for bigram in list(product(alfabeto, repeat=2))]
alfabeto_bigrams

PASSO 3) CAPTURA DOS FALSOS POSITIVOS DO MODELO 2 NO VETOR DE ENTRADA DO MODELO 2

A partir do vetor de bigramas podemos identificar uma palavra. Logo, o que foi feito? 

1) Primeiramente a realizaçao do vetor de bigramas de cada falso positivo encontrado.

2) Verificação da existencia destes vetores dentro do vetor de entrada que o modelo utiliza.

Neste vetor de entrada, as informaçoes sobre os predecessores e sucessores existem. Logo, a captura dessas informações conterão os mesmos estados em que os falsos positivos se encontram dentro da base.

In [ ]:
df_result = tokens_vector_bigrams.merge(test_vector_input_model, on=alfabeto_bigrams, how='left')
df_result

Visto que o vetor de bigramas contem todas as ocorrencias dos falsos positivos, o número do datarame é superior ao número de tokens obtidos a partir da matriz de confusão.

Removendo a duplicidade do vetor de bigramas, podemos obter o número real de tokens que existem.

Após essa etapa, podemos identificar que sim, se tratam dos 265 tokens extraídos da matriz de confusao

In [ ]:
df_result.drop_duplicates(subset=alfabeto_bigrams).shape

Para verificar se se trata realmente dos 265 tokens ( falsos positivos novos )

In [ ]:
df_result.drop_duplicates(subset=alfabeto_bigrams).shape

#### Estes tokens possuem vizinhos que são nomes?

Dos 265 tokens - 96 possuem ambos os vizinhos nomes ( tanto predecessor quando sucessor )

In [ ]:
pred_n_suce_n = df_result[(df_result['y_predecessor'] == 1) & (df_result['y_sucessor'] == 1)]
pred_n_suce_n.drop_duplicates(subset=alfabeto_bigrams)

Também entre estes tokens ( Os 265 ) 223 possuem ao menos um nome como vizinho

In [ ]:
pred_nw_suce_nw = df_result[(df_result['y_predecessor'] == 1) | (df_result['y_sucessor'] == 1)]
pred_nw_suce_nw.drop_duplicates(subset=alfabeto_bigrams)

Os numeros não batem, né? se somar. O motivo é simples, um mesmo token pode estar em uma posiçao.

y_p = 0 | y_s = 0  
y_p = 1 | y_s = 0  
  
y_p = 0 | y_s = 1  
y_p = 1 | y_s = 1  

In [ ]:
pred_w_suce_w = df_result[(df_result['y_predecessor'] == 0) & (df_result['y_sucessor'] == 0)]
pred_w_suce_w.drop_duplicates(subset=alfabeto_bigrams)

Um mesmo tokem pode possuir classificaçoes de predecessores e sucessores diferentes nas bases?

In [ ]:
pred_nw_suce_nw = pred_nw_suce_nw.drop_duplicates(subset=alfabeto_bigrams)
pred_w_suce_w = pred_w_suce_w.drop_duplicates(subset=alfabeto_bigrams)

In [ ]:
intecessao = pred_w_suce_w.merge(pred_nw_suce_nw, on=alfabeto_bigrams, how='inner')
intecessao

In [ ]:
v = pred_w_suce_w.merge(pred_nw_suce_nw, on=alfabeto_bigrams, how='inner')

if not v.empty:
    print("Existe interseção")
else:
    print("Não existe interseção")


#### Explicabilidade dos Falsos Positivos

In [ ]:
rf_corretor = load('rf_corretor.joblib')

In [ ]:
df_tests = df_result.drop(columns=['palavra', 'predecessor', 'sucessor'])
df_tests

In [ ]:
importancia_features = rf_corretor.feature_importances_

feature_names = df_tests.columns 
importancia_df = pd.DataFrame({
    'Feature': feature_names,
    'Importância': importancia_features
}).sort_values(by='Importância', ascending=False)

importancia_df


Uma análise em cima apenas dos dados que foram preditos como 1. Ou seja o que os determinou em irem para a categoria de Falsos Positivos

captura dos dados que foram preditos como falsos positivos

In [ ]:
fp_pred_one_m2 = pd.DataFrame({'pred' : rf_corretor.predict(df_tests)})
fp_pred_one_m2[fp_pred_one_m2['pred']==1]

In [ ]:
fp_pred_one_m2 = pd.DataFrame({'pred': rf_corretor.predict(df_tests)})
filtered_df_fp = df_tests[fp_pred_one_m2['pred'] == 1]

filtered_df_fp

In [ ]:
pred_n_suce_n = filtered_df_fp[(filtered_df_fp['y_predecessor'] == 1) & (filtered_df_fp['y_sucessor'] == 1)]
pred_n_suce_n.drop_duplicates(subset=alfabeto_bigrams).shape

In [ ]:
pred_nw_suce_nw = filtered_df_fp[(filtered_df_fp['y_predecessor'] == 1) | (filtered_df_fp['y_sucessor'] == 1)]
pred_nw_suce_nw.drop_duplicates(subset=alfabeto_bigrams).shape

Esses caras aqui, são os nossos objetos de estudo:

Esses fogem a regra de predecessores e sucessores nomes.

In [ ]:
pred_w_suce_w = filtered_df_fp[(filtered_df_fp['y_predecessor'] == 0) & (filtered_df_fp['y_sucessor'] == 0)]
pred_w_suce_w.drop_duplicates(subset=alfabeto_bigrams).shape

In [ ]:
pred_w_suce_w

0   - ADJETIVO  
1   - ADVERBIO  
2   - ARTIGO  
3   - CONJUNCAO  
4   - ESPACO  
5   - NULL  
6   - NUMERAL  
7   - PALAVRA ESTRANGEIRA  
8   - PONTUACAO  
9   - PREPOSIÇÃO  
10  - PRONOME  
11  - SIMBOLO  
12  - SUBSTANTIVO  
13  - VERBO  

dos 59 tokens, possuímos 102 ocorrências

In [ ]:
import lime
import lime.lime_tabular

predict_fn_rf = lambda x: rf_corretor.predict_proba(x).astype(float)

X = pred_w_suce_w.values

explainer = lime.lime_tabular.LimeTabularExplainer(X, feature_names=pred_w_suce_w.columns)

choosen_instance = pred_w_suce_w.values[101]

exp = explainer.explain_instance(choosen_instance, predict_fn_rf, num_features=10)

exp.show_in_notebook(show_all=False)


In [ ]:
df_result

In [ ]:
pred_w_suce_w

In [ ]:
valores_comuns = df_result[alfabeto_bigrams].isin(pred_w_suce_w)
valores_comuns

In [ ]:
presentes = df_result[valores_comuns.any(axis=1)]
presentes

In [ ]:
df_tratado = presentes.drop(columns=['predecessor', 'palavra', 'sucessor'])

In [ ]:
import lime
import lime.lime_tabular

predict_fn_rf = lambda x: rf_corretor.predict_proba(x).astype(float)

X = df_tratado.values

explainer = lime.lime_tabular.LimeTabularExplainer(X, feature_names=df_tratado.columns)

choosen_instance = df_tratado.values[101]

exp = explainer.explain_instance(choosen_instance, predict_fn_rf, num_features=10)

exp.show_in_notebook(show_all=False)
